# 03d — CP2 v2 production parity / refreeze gate

Purpose: verify that the production CP2-v2 implementation reproduces the already-audited Notebook-03 all-resolved-timezone candidate on the frozen **5,821 CP1 stays**.

This notebook does **not** rescan raw `.plt` files and does **not** rerun timezone/geography/DBSCAN sensitivity research. It rebuilds only the small independent Notebook-03 reference from the frozen stay cache, then compares production by stay scope, timezone/local wall time, complete-link cluster membership, and HOME/OFFICE selected clusters.

Important: raw sklearn `location_id` integers are not used as parity keys because Notebook 03 and production intentionally label equivalent clusters differently. Parity uses the exact stay membership of each cluster.


In [ ]:
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp2-v2-timezone-production")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
PARITY_DIR = VOLUME_ROOT / "cache" / "cp2_v2_parity"
PARITY_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

ensure_repo()
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR)

for candidate in (REPO_DIR, REPO_DIR / "src"):
    value = str(candidate)
    if value not in sys.path:
        sys.path.insert(0, value)

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering
from timezonefinder import TimezoneFinder

print("Repo branch:", REPO_BRANCH)
print("Repo SHA:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())
print("Parity dir:", PARITY_DIR)


In [ ]:
def find_stay_cache():
    candidates = [
        VOLUME_ROOT / "cache" / "cp2_home_office" / "stays_baseline_v1.pkl",
        VOLUME_ROOT / "cache" / "03a_user_behavior_deep_dive" / "stays_baseline_v1.pkl",
        VOLUME_ROOT / "artifacts" / "03a" / "stays_baseline_v1.pkl",
        REPO_DIR / "artifacts" / "03a" / "stays_baseline_v1.pkl",
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob("**/stays_baseline_v1.pkl"))
    if matches:
        return matches[0]
    raise FileNotFoundError("Cannot find frozen stays_baseline_v1.pkl")

STAYS_CACHE = find_stay_cache()
stays = pd.read_pickle(STAYS_CACHE)
stays["user_id"] = stays["user_id"].astype(str)

assert len(stays) == 5_821, len(stays)
assert stays["user_id"].nunique() == 136, stays["user_id"].nunique()

print("Frozen cache:", STAYS_CACHE)
print("CP1 stays:", len(stays))
print("CP1 stay users:", stays["user_id"].nunique())


## Independent Notebook-03 reference

The code below mirrors the **final timezone-v2 candidate path** from `03_home_office_baseline.ipynb` rather than importing `build_semantic_locations()` or `infer_home_office()` from production. This keeps the parity check non-circular.


In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LOCATION_THRESHOLD_M = 200.0
HOME_NIGHT_START_HOUR = 21
HOME_NIGHT_END_HOUR = 6
OFFICE_START_HOUR = 9
OFFICE_END_HOUR = 17
OFFICE_WEEKDAYS = {0, 1, 2, 3, 4}
MIN_RELEVANT_DATE_OVERLAP_S = 10 * 60

finder = TimezoneFinder()
reference_stays = stays.copy()
reference_stays["timezone_id"] = [
    finder.timezone_at(lng=float(lon), lat=float(lat))
    for lat, lon in zip(reference_stays["latitude"], reference_stays["longitude"], strict=True)
]
reference_stays = reference_stays.loc[reference_stays["timezone_id"].notna()].copy()

def utc_to_local_wall_time(timestamp_utc, timezone_id):
    ts = pd.Timestamp(timestamp_utc)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.tz_convert(ZoneInfo(str(timezone_id))).tz_localize(None)

reference_stays["arrival_time_local"] = pd.to_datetime([
    utc_to_local_wall_time(ts, tzid)
    for ts, tzid in zip(reference_stays["arrival_time_utc"], reference_stays["timezone_id"], strict=True)
])
reference_stays["departure_time_local"] = pd.to_datetime([
    utc_to_local_wall_time(ts, tzid)
    for ts, tzid in zip(reference_stays["departure_time_utc"], reference_stays["timezone_id"], strict=True)
])
reference_stays["arrival_local_date"] = reference_stays["arrival_time_local"].dt.date

assert len(reference_stays) == 5_821, "Notebook-03 audit expected every frozen stay to resolve"

def pairwise_haversine_matrix_m(group):
    lat = np.radians(group["latitude"].to_numpy(dtype=float))
    lon = np.radians(group["longitude"].to_numpy(dtype=float))
    dlat = lat[:, None] - lat[None, :]
    dlon = lon[:, None] - lon[None, :]
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_M * np.arcsin(np.minimum(1.0, np.sqrt(a)))

clustered_parts = []
location_rows = []
for user_id, group in reference_stays.groupby("user_id", sort=True):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    if len(g) == 1:
        distances = np.zeros((1, 1), dtype=float)
        labels = np.zeros(1, dtype=int)
    else:
        distances = pairwise_haversine_matrix_m(g)
        labels = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="complete",
            distance_threshold=LOCATION_THRESHOLD_M,
        ).fit_predict(distances)
    g["location_id"] = labels.astype(int)
    clustered_parts.append(g)

    for location_id in np.unique(labels):
        member_idx = np.flatnonzero(labels == location_id)
        members = g.iloc[member_idx]
        diameter_m = float(distances[np.ix_(member_idx, member_idx)].max()) if len(member_idx) > 1 else 0.0
        location_rows.append({
            "user_id": user_id,
            "location_id": int(location_id),
            "latitude": float(members["latitude"].median()),
            "longitude": float(members["longitude"].median()),
            "stay_count": int(len(members)),
            "active_local_dates": int(members["arrival_local_date"].nunique()),
            "total_dwell_h": float(members["duration_s"].sum() / 3600.0),
            "diameter_m": diameter_m,
        })

reference_semantic_stays = pd.concat(clustered_parts, ignore_index=True)
reference_locations = pd.DataFrame(location_rows)
assert reference_locations["diameter_m"].max() <= LOCATION_THRESHOLD_M + 1e-6

print("Reference timezone-resolved stays:", len(reference_semantic_stays))
print("Reference semantic users:", reference_semantic_stays["user_id"].nunique())
print("Reference semantic locations:", len(reference_locations))
print("Reference recurring users:", reference_locations.loc[reference_locations["stay_count"].ge(2), "user_id"].nunique())


In [ ]:
def interval_overlap_s(start, end, window_start, window_end):
    overlap_start = max(start, window_start)
    overlap_end = min(end, window_end)
    if overlap_end <= overlap_start:
        return 0.0
    return float((overlap_end - overlap_start).total_seconds())

night_rows = []
office_rows = []
for row in reference_semantic_stays.itertuples(index=False):
    start = row.arrival_time_local
    end = row.departure_time_local
    day = start.normalize() - pd.Timedelta(days=1)
    last_day = end.normalize()
    while day <= last_day:
        night_start = day + pd.Timedelta(hours=HOME_NIGHT_START_HOUR)
        night_end = day + pd.Timedelta(days=1, hours=HOME_NIGHT_END_HOUR)
        night_s = interval_overlap_s(start, end, night_start, night_end)
        if night_s > 0:
            night_rows.append({"user_id": row.user_id, "location_id": int(row.location_id), "behavior_date": night_start.date(), "overlap_s": night_s})

        if day.weekday() in OFFICE_WEEKDAYS:
            office_start = day + pd.Timedelta(hours=OFFICE_START_HOUR)
            office_end = day + pd.Timedelta(hours=OFFICE_END_HOUR)
            office_s = interval_overlap_s(start, end, office_start, office_end)
            if office_s > 0:
                office_rows.append({"user_id": row.user_id, "location_id": int(row.location_id), "behavior_date": office_start.date(), "overlap_s": office_s})
        day += pd.Timedelta(days=1)

def aggregate_window(rows, prefix):
    contrib = pd.DataFrame(rows, columns=["user_id", "location_id", "behavior_date", "overlap_s"])
    if contrib.empty:
        return pd.DataFrame(columns=["user_id", "location_id", f"{prefix}_dwell_s", f"{prefix}_dates"])
    per_date = contrib.groupby(["user_id", "location_id", "behavior_date"], as_index=False)["overlap_s"].sum()
    dwell = per_date.groupby(["user_id", "location_id"], as_index=False)["overlap_s"].sum().rename(columns={"overlap_s": f"{prefix}_dwell_s"})
    dates = per_date.loc[per_date["overlap_s"].ge(MIN_RELEVANT_DATE_OVERLAP_S)].groupby(["user_id", "location_id"], as_index=False)["behavior_date"].nunique().rename(columns={"behavior_date": f"{prefix}_dates"})
    return dwell.merge(dates, on=["user_id", "location_id"], how="left").fillna({f"{prefix}_dates": 0})

features = reference_locations.merge(aggregate_window(night_rows, "night"), on=["user_id", "location_id"], how="left").merge(aggregate_window(office_rows, "office"), on=["user_id", "location_id"], how="left")
for column in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
    features[column] = features[column].fillna(0)
features["night_dates"] = features["night_dates"].astype(int)
features["office_dates"] = features["office_dates"].astype(int)
night_total = features.groupby("user_id")["night_dwell_s"].transform("sum")
office_total = features.groupby("user_id")["office_dwell_s"].transform("sum")
features["night_dwell_share"] = np.where(night_total > 0, features["night_dwell_s"] / night_total, 0.0)
features["office_dwell_share"] = np.where(office_total > 0, features["office_dwell_s"] / office_total, 0.0)

def top_with_gate(label, min_dates, min_share, min_margin):
    if label == "HOME":
        share_col, dates_col, dwell_col = "night_dwell_share", "night_dates", "night_dwell_s"
    else:
        share_col, dates_col, dwell_col = "office_dwell_share", "office_dates", "office_dwell_s"
    ranked = features.loc[(features["stay_count"].ge(2)) & (features[dates_col].ge(min_dates)) & (features[dwell_col].gt(0))].copy()
    ranked = ranked.sort_values(["user_id", share_col, dates_col, dwell_col, "stay_count"], ascending=[True, False, False, False, False], kind="stable")
    ranked["rank"] = ranked.groupby("user_id").cumcount() + 1
    top = ranked.loc[ranked["rank"].eq(1)].copy()
    second = ranked.loc[ranked["rank"].eq(2), ["user_id", share_col]].rename(columns={share_col: "second_share"})
    top = top.merge(second, on="user_id", how="left")
    top["second_share"] = top["second_share"].fillna(0.0)
    top["share_margin"] = top[share_col] - top["second_share"]
    emitted = top.loc[top[share_col].ge(min_share) & top["share_margin"].ge(min_margin)].copy()
    emitted["label"] = label
    emitted["relevant_dwell_share"] = emitted[share_col]
    emitted["relevant_dates"] = emitted[dates_col].astype(int)
    emitted["evidence_strength"] = (emitted["relevant_dwell_share"] + emitted["share_margin"] + np.minimum(emitted["relevant_dates"] / 5.0, 1.0)) / 3.0
    return emitted

reference_home = top_with_gate("HOME", 3, 0.50, 0.20)
reference_office = top_with_gate("OFFICE", 3, 0.30, 0.10)
reference_labels = pd.concat([reference_home, reference_office], ignore_index=True)

REFERENCE_STAYS = PARITY_DIR / "notebook03_semantic_stays.pkl"
REFERENCE_LOCATIONS = PARITY_DIR / "notebook03_semantic_locations.pkl"
REFERENCE_LABELS = PARITY_DIR / "notebook03_labels.pkl"
reference_semantic_stays.to_pickle(REFERENCE_STAYS)
reference_locations.to_pickle(REFERENCE_LOCATIONS)
reference_labels.to_pickle(REFERENCE_LABELS)

display(pd.Series({
    "semantic_users": reference_semantic_stays["user_id"].nunique(),
    "semantic_locations": len(reference_locations),
    "recurring_users": reference_locations.loc[reference_locations["stay_count"].ge(2), "user_id"].nunique(),
    "recurring_locations": int(reference_locations["stay_count"].ge(2).sum()),
    "home_emitted": len(reference_home),
    "office_emitted": len(reference_office),
    "unique_emitted_users": reference_labels["user_id"].nunique(),
}, name="notebook03_reference").to_frame())


In [ ]:
RESULT_PATH = PARITY_DIR / "parity_result.json"
run([
    sys.executable,
    str(REPO_DIR / "analysis" / "03_cp2_v2_parity.py"),
    "--stays", str(STAYS_CACHE),
    "--reference-semantic-stays", str(REFERENCE_STAYS),
    "--reference-locations", str(REFERENCE_LOCATIONS),
    "--reference-labels", str(REFERENCE_LABELS),
    "--output", str(RESULT_PATH),
], cwd=REPO_DIR)

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
display(pd.DataFrame([result["production_v2"], result["notebook03_reference"]], index=["production_v2", "notebook03_reference"]))
display(pd.Series(result["notebook03_parity"], name="parity" ).to_frame())

assert result["all_input_stays_timezone_resolved"] is True
assert result["notebook03_parity"]["all_parity_checks_pass"] is True, result["notebook03_parity"]

print("CP2-v2 PARITY PASS")
print("Safe to proceed to the CP2-v2 namespace refreeze decision.")
